<a href="https://colab.research.google.com/github/grasht/projects_ML_HW_5/blob/main/HW_5_tsk_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 3 NLP and Attention Mechanism

# Part 1 Scaled Dot-Product Attention

In [93]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [94]:
import numpy as np
import pandas as pd

#Scaled Dot Product Attention
def sdp_attention(query, key, value):
  def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)

  QK = query @ key.T
  dk = key.shape[1]
  scaledQK = QK/np.sqrt(dk)
  softmaxQK = softmax(scaledQK)
  return softmaxQK @ value, softmaxQK



# Part 2 Encoder-Decoder Model with Integrated Attention Mechanism



In [95]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)

        #I tried this with LSTM first, but when I realized we are using a small-scale
        #dataset in Part3 I switched to GRU
        #self.rnn = nn.LSTM(emb_dim, hid_dim, batch_first=True)

        self.rnn = nn.GRU(emb_dim, hid_dim, batch_first=True)

    def forward(self, src):
        embedded = self.embedding(src)

        # outputs: (batch, seq_len, hidden_dim)
        # hidden: (1, batch, hidden_dim)
        outputs, hidden = self.rnn(embedded)

        return outputs, hidden

In [96]:
class Attention(nn.Module):
  def __init__(self):
    super().__init__()

  #Copy this into the Class definition to avoid relying on a global function
  # def sdp_attention(query, key, value):
  #   def softmax(x):
  #     e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
  #     return e_x / np.sum(e_x, axis=-1, keepdims=True)

  #   QK = query @ key.T
  #   dk = key.shape[1]
  #   scaledQK = QK/np.sqrt(dk)
  #   softmaxQK = softmax(scaledQK)
  #   return softmaxQK @ value, softmaxQK

  #Need to reimplement sdp_attention using pytorch or it wont work with tensors
  def sdp_attention(self, query, key, value):
    dk = key.size(-1)
    scores = torch.bmm(query, key.transpose(1, 2)) / torch.sqrt(torch.tensor(dk, dtype=torch.float, device=key.device))
    attn_weights = torch.softmax(scores, dim=-1)
    context = torch.bmm(attn_weights, value)
    return context, attn_weights

  def forward(self, decoder_hidden, encoder_outputs):
    query = decoder_hidden[-1].unsqueeze(1)
    print(query.shape)
    key = encoder_outputs
    value = encoder_outputs

    context, attn_weights = self.sdp_attention(query, key, value)

    return context, attn_weights

In [97]:
class Decoder(nn.Module):
  def __init__(self, output_dim, emb_dim, hid_dim):
    super().__init__()

    self.embedding = nn.Embedding(output_dim, emb_dim)
    #self.rnn = nn.LSTM(emb_dim + hid_dim, hid_dim, batch_first=True)
    self.rnn = nn.GRU(emb_dim + hid_dim, hid_dim, batch_first=True)

    self.fc = nn.Linear(hid_dim *2, output_dim)

    self.attention = Attention()

  def forward(self, input, hidden, encoder_outputs):
    input = input.unsqueeze(1)
    embedded = self.embedding(input)

    context, attn_weights = self.attention(hidden, encoder_outputs)
    rnn_input = torch.cat((embedded, context), dim=2)

    output, hidden = self.rnn(rnn_input, hidden)

    output = output.squeeze(1)
    context = context.squeeze(1)

    prediction = self.fc(torch.cat((output, context), dim=1))

    return prediction, hidden, attn_weights

In [98]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size, trg_len = trg.shape
        trg_vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size)

        encoder_outputs, hidden = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, _ = self.decoder(input, hidden, encoder_outputs)
            outputs[:, t] = output

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            top1 = output.argmax(1)

            input = trg[:, t] if teacher_force else top1

        return outputs

# Part 3 Pick a Data Set and Train the Model

In [99]:
# Load dataset
from datasets import load_dataset

dataset = load_dataset("bentrevett/multi30k")

def tokenize_de(text):
    return text.lower().split()

def tokenize_en(text):
    return text.lower().split()

In [100]:
def train(model, dataloader, optimizer, criterion, clip, device):
    model.train()
    epoch_loss = 0

    for src, trg in dataloader:
        src = src.to(device)   # German
        trg = trg.to(device)   # English

        optimizer.zero_grad()

        # output: [trg_len, batch_size, output_dim]
        output = model(src, trg, teacher_forcing_ratio=0.5)

        # Ignore <sos>
        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim)
        trg = trg[1:].reshape(-1)

        # Compute loss
        loss = criterion(output, trg)

        # Backprop
        loss.backward()

        # Gradient clipping (important for RNNs)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [101]:
def evaluate(model, dataloader, criterion, device):

    model.eval()
    epoch_loss = 0

    with torch.no_grad():

        for src, trg in dataloader:

            src = src.to(device)
            trg = trg.to(device)

            # No teacher forcing
            output = model(src, trg, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]

            output = output[1:].view(-1, output_dim)
            trg = trg[1:].reshape(-1)

            loss = criterion(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

In [102]:
from collections import Counter

def build_vocab(sentences, tokenizer, min_freq=2):
    counter = Counter()
    for sentence in sentences:
        counter.update(tokenizer(sentence))

    vocab = {
        "<pad>": 0,
        "<sos>": 1,
        "<eos>": 2,
        "<unk>": 3
    }

    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)

    return vocab

vocab_de = build_vocab(dataset["train"]["de"], tokenize_de)
vocab_en = build_vocab(dataset["train"]["en"], tokenize_en)

In [103]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    src_batch = []
    trg_batch = []

    for src, trg in batch:
        src_tensor = torch.tensor(process(src, vocab_de, tokenize_de), dtype=torch.long)
        trg_tensor = torch.tensor(process(trg, vocab_en, tokenize_en), dtype=torch.long)

        src_batch.append(src_tensor)
        trg_batch.append(trg_tensor)

    # Pad sequences
    src_batch = pad_sequence(src_batch, batch_first=True, padding_value=PAD_IDX)
    trg_batch = pad_sequence(trg_batch, batch_first=True, padding_value=PAD_IDX)

    return src_batch, trg_batch

In [104]:
from torch.utils.data import Dataset

class TranslationDataset(Dataset):
    def __init__(self, hf_dataset, src_lang="de", trg_lang="en"):
        self.src_data = hf_dataset[src_lang]
        self.trg_data = hf_dataset[trg_lang]

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.trg_data[idx]

In [105]:
from torch.utils.data import DataLoader

train_dataset = TranslationDataset(dataset["train"])
valid_dataset = TranslationDataset(dataset["validation"])
test_dataset  = TranslationDataset(dataset["test"])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [106]:
PAD_IDX = vocab_en["<pad>"]

def numericalize(sentence, vocab, tokenizer):
    tokens = tokenizer(sentence)
    return [vocab.get(token, vocab["<unk>"]) for token in tokens]

def process(sentence, vocab, tokenizer):
    return [vocab["<sos>"]] + \
           numericalize(sentence, vocab, tokenizer) + \
           [vocab["<eos>"]]

for batch in train_loader:
    src, trg = batch
    src = src.to(device)
    trg = trg.to(device)


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
model = Seq2Seq(Encoder(len(vocab_de), 256, 512), Decoder(len(vocab_en), 256, 512))
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters())

N_EPOCHS = 10
CLIP = 1.0

for epoch in range(N_EPOCHS):

    train_loss = train(model, train_loader, optimizer, criterion, CLIP, device)
    valid_loss = evaluate(model, valid_loader, criterion, device)

    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {train_loss:.3f}")
    print(f"Val Loss:   {valid_loss:.3f}")

Streaming output truncated to the last 5000 lines.
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])
torch.Size([32, 1, 512])

# Part 4 Simplified Transformer Model